## tl;dr

SquadMetric's reconstructed, strict walk-forward 2026/27 run has **179 points through finalized GW3**. The official FPL averages sum to 182, so the model is 3 points below average overall. It scored 36 in GW1, 83 in GW2, and 60 in GW3.

## Context & Methods

This notebook calls the repository's deterministic current-season replay. For each gameweek, the points and minutes models are refitted using completed historical seasons plus only 2026/27 rows from earlier gameweeks. The squad, lineup, captain, and transfer decision use the archived FPL bootstrap and fixture snapshot captured before that gameweek's deadline. Actual points are joined only after the decision to score the team.

The opening squad is optimized from the pre-GW1 snapshot with an eight-gameweek horizon. A free transfer requires at least 2.0 projected horizon points; a minus-four hit therefore requires at least 6.0 gross projected horizon points. Chips are evaluated jointly with transfers at every legal deadline. Automatic actions on a real FPL account remain disabled.

### Key Assumptions

- Published future fixtures are allowed because they were public before each deadline.
- GW4 is excluded until official FPL marks it finished and data-checked.
- Per-deadline ranked-player role overlays were not archived, so availability comes from each deadline bootstrap and the minutes model is refitted point-in-time.
- This is a reconstruction of the current corrected SquadMetric policy, not proof of the exact output emitted by an older deployed build.

## Data

The inputs are the repository's historical player-gameweek table, finalized live 2026/27 player-gameweek rows, and hash-matched pre-deadline FPL snapshots.

In [1]:
import pandas as pd
from IPython.display import display

from fpl_intelligence.current_season_replay import run_current_season_replay

report = run_current_season_replay()
assert report['leakage_checks']['target_or_future_results_in_training'] is False
assert sum(row['net_points'] for row in report['gameweeks']) == report['total_points']
assert report['finalized_gameweeks'] == [1, 2, 3]

source_columns = ['gameweek', 'captured_at', 'deadline', 'bootstrap_hash']
display(pd.DataFrame(report['sources'])[source_columns])

,gameweek,captured_at,deadline,bootstrap_hash
0,1,2026-08-18T06:11:54.361356Z,2026-08-21T17:30:00Z,dc2e8ca5c7976b69ba3fc0b35a083900a2fb2a54d40238...
1,2,2026-08-27T17:32:57.469898Z,2026-08-28T17:30:00Z,2c1b9c17efb2a8383f06f680cc3c1bba8aa6322745b232...
2,3,2026-09-01T17:41:58.186765Z,2026-09-04T17:30:00Z,b85dbac201547f4320cea732a0ee8a836baf545ea05783...


## Results

In [2]:
weekly = pd.DataFrame(report['gameweeks'])
weekly['transfer_summary'] = weekly['transfers'].apply(
    lambda moves: ', '.join(f"{move['out']} -> {move['in']}" for move in moves) or 'Roll'
)
summary_columns = [
    'gameweek', 'net_points', 'cumulative_points', 'official_average',
    'delta_vs_average', 'transfer_summary', 'transfer_projected_horizon_gain',
    'hit_cost', 'chip', 'captain', 'formation'
]
display(weekly[summary_columns])
print(f"SquadMetric total: {report['total_points']:.0f}")
print(f"Official-average total: {report['official_average_total']:.0f}")
print(f"Difference: {report['delta_vs_official_average']:+.0f}")

,gameweek,net_points,cumulative_points,official_average,delta_vs_average,transfer_summary,transfer_projected_horizon_gain,hit_cost,chip,captain,formation
0,1,36.0,36.0,50,-14.0,Roll,0.00,0,3xc,Erling Haaland,3-4-3
1,2,83.0,119.0,81,2.0,Ollie Watkins -> João Pedro Junqueira de Jesus...,26.49,4,bboost,Erling Haaland,3-4-3
2,3,60.0,179.0,51,9.0,Senne Lammens -> Konstantinos Tzolakis,5.13,0,NaN,Erling Haaland,3-4-3


SquadMetric total: 179
Official-average total: 182
Difference: -3


In [3]:
opening_team = pd.DataFrame(report['initial_squad']).sort_values(
    ['position', 'price'], ascending=[True, False]
)
display(opening_team[['position', 'player_name', 'price']].reset_index(drop=True))
print(f"Opening spend: {opening_team['price'].sum():.1f}m; bank: {report['initial_bank']:.1f}m")

,position,player_name,price
0,DEF,Marc Guéhi,6.0
1,DEF,Adam Smith,4.5
2,DEF,Luke Shaw,4.5
3,DEF,Issa Diop,4.0
4,DEF,Luke O'Nien,4.0
5,FWD,Erling Haaland,15.5
6,FWD,Ollie Watkins,8.0
7,FWD,Dominic Calvert-Lewin,6.0
8,GKP,Senne Lammens,5.0
9,GKP,Antonín Kinský,4.5


Opening spend: 100.0m; bank: 0.0m


## Takeaways

The opening week still did the damage: Triple Captain on Haaland produced 36 points, 14 below the GW1 average. In GW2 the model used Bench Boost and took a justified minus-four for Watkins-to-Joao Pedro and Smith-to-De Cuyper; the transfer pair had a projected 26.49-point horizon gain. In GW3 it used a free transfer on Lammens-to-Tzolakis. It is 3 points below the cumulative official average.